# Customer Lifetime Value Prediction & Customer Churn Survival Analysis
**Projet Data Mining / Machine Learning — Online Retail II (UCI)**

## 1. Introduction

**Message de synthèse du projet :**
> À partir de l'historique d'achat des clients, nous prédisons leur revenu futur, estimons leur probabilité de rester actifs grâce à la Survival Analysis, calculons leur Customer Lifetime Value (CLV) et les segmentons selon leur valeur et leur risque de churn afin de proposer des actions marketing adaptées.

### Structure du notebook (pipeline)

```
```

| # | Section | # | Section |
|---|---|---|---|
| 1 | Introduction | 13 | Model Comparison |
| 2 | Business Problem | 14 | Survival Analysis |
| 3 | Dataset Description | 15 | Kaplan-Meier |
| 4 | Import Libraries | 16 | Cox Proportional Hazards |
| 5 | Data Loading | 17 | Weibull Survival Model |
| 6 | Data Cleaning | 18 | CLV Calculation |
| 7 | Exploratory Data Analysis | 19 | Customer Segmentation |
| 8 | Customer-Level Dataset | 20 | Marketing Recommendations |
| 9 | RFM Analysis | 21 | Visualizations |
| 10 | Feature Engineering | 22 | Business Insights |
| 11 | Temporal Split | 23 | Conclusion |
| 12 | Future Revenue Prediction | | (+ export Dashboard Power BI) |

> **Règle d'or du notebook :** aucun résultat n'est écrit à la main. Tous les chiffres sont produits par le code ; les interprétations sont des grilles de lecture que tu complèteras avec les valeurs obtenues à l'exécution.

## 2. Business Problem

| Question métier | Réponse apportée par |
|---|---|
| Q1 — Quels sont les meilleurs clients ? | RFM + CLV |
| Q2 — Quels clients risquent de partir ? | Survival Analysis (Cox / Weibull) → `Churn_Risk` |
| Q3 — Combien chaque client rapportera-t-il ? | Régression sur `Future_Revenue` |
| Q4 — Quelle est sa valeur totale future ? | `Predicted_CLV` |
| Q5 — Qui cibler en priorité ? | Segmentation CLV × Churn + `Priority_Score` |

**Définition du churn :** un client a churné s'il n'a fait **aucun achat pendant 180 jours** (seuils 90 et 270 jours testés en sensibilité).
**Marge supposée :** 30 %.

## 3. Dataset Description

**Online Retail II (UCI)** : transactions d'un e-commerce britannique (cadeaux/articles de maison, beaucoup de clients grossistes), du 01/12/2009 au 09/12/2011.

| Colonne | Signification |
|---|---|
| `Invoice` | N° de facture (préfixe **C** = annulation) |
| `StockCode` | Code produit |
| `Description` | Libellé produit |
| `Quantity` | Quantité (négative pour les retours) |
| `InvoiceDate` | Date/heure de la facture |
| `Price` | Prix unitaire (£) |
| `Customer ID` | Identifiant client (souvent manquant) |
| `Country` | Pays du client |

Le fichier Excel contient **deux feuilles** (2009-2010 et 2010-2011) qu'on concatène.

## 4. Import Libraries

In [ ]:
!pip -q install lifelines xlrd openpyxl xgboost

import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score

from lifelines import KaplanMeierFitter, CoxPHFitter, WeibullAFTFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from scipy.stats import spearmanr

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)

# ---------------- Paramètres du projet (modifiables) ----------------
RANDOM_STATE   = 42
MARGIN         = 0.30          # marge bénéficiaire supposée
CHURN_DAYS     = 180           # définition principale du churn
THRESHOLDS     = [90, 180, 270]  # seuils testés en sensibilité
HORIZON        = 180           # horizon de prédiction du revenu futur (jours)
SURV_FOLLOWUP  = 365           # suivi du landmark survie (jours)
CLV_PERIODS    = 6             # nb de périodes de 180 jours dans la CLV (= 3 ans)
DISCOUNT_RATE  = 0.10          # taux d'actualisation annuel
CLV_HIGH_Q     = 0.75          # High CLV = top 25 %
RISK_THRESHOLD = 0.50          # High Risk si Churn_Risk >= 50 %
DROP_DUPLICATES = True         # supprimer les lignes strictement dupliquées
REMOVE_NON_PRODUCT = True      # supprimer POST, DOT, BANK CHARGES, ADJUST... (non-produits)

## 5. Data Loading

Le code télécharge le fichier depuis l'UCI. **Si le téléchargement échoue** dans Colab, télécharge `online_retail_II.xlsx` manuellement (UCI ou Kaggle) et uploade-le dans le panneau *Fichiers* de Colab : le code le détectera.

In [ ]:
if "df_raw" not in globals():
    if not os.path.exists("online_retail_II.xlsx"):
        URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        !wget -q -O online_retail_ii.zip "{URL}"
        !unzip -o -q online_retail_ii.zip
    sheets = pd.read_excel("online_retail_II.xlsx", sheet_name=None)
    df_raw = pd.concat(sheets.values(), ignore_index=True)

print("Dimensions :", df_raw.shape)
display(df_raw.head())
print("\nTypes de variables :")
print(df_raw.dtypes)
print("\nValeurs manquantes :")
miss = pd.DataFrame({"manquantes": df_raw.isna().sum(), "%": df_raw.isna().mean() * 100})
display(miss)
print(f"\nNombre de clients distincts (Customer ID non nul) : {df_raw['Customer ID'].nunique():,}")
print(f"Période : {df_raw['InvoiceDate'].min()}  →  {df_raw['InvoiceDate'].max()}")
print(f"Lignes dupliquées exactes : {df_raw.duplicated().sum():,}")
print(f"Lignes d'annulation (Invoice commençant par 'C') : {df_raw['Invoice'].astype(str).str.startswith('C').sum():,}")
print("\nDistribution de Quantity et Price :")
display(df_raw[["Quantity", "Price"]].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
print(f"Quantity <= 0 : {(df_raw['Quantity'] <= 0).sum():,}  |  Price <= 0 : {(df_raw['Price'] <= 0).sum():,}")

**Premier diagnostic — à lire dans les sorties ci-dessus :**
- Une part importante des lignes n'a **pas de `Customer ID`** → inutilisables pour une analyse *par client*.
- `Quantity` et `Price` ont des valeurs ≤ 0 (retours, ajustements comptables) et des **queues très épaisses** (min/max extrêmes vs. médiane) : attendre une distribution très asymétrique du revenu.
- Les annulations (`Invoice` commençant par `C`) et les doublons exacts doivent être traités avant toute agrégation.

## 6. Data Cleaning

Règles appliquées, dans cet ordre, **avec le nombre de lignes supprimées à chaque étape** :

1. (optionnel) doublons exacts ;
2. annulations / retours (`Invoice` commençant par `C`) ;
3. `Customer ID` manquant ;
4. `Quantity` ≤ 0 ;
5. `Price` ≤ 0 ;
6. (optionnel) lignes non-produits (frais de port `POST`, `BANK CHARGES`, `ADJUST`, `M`, `DOT`, etc.) : ce ne sont pas des achats de produits et ils faussent le panier moyen. Désactivable via `REMOVE_NON_PRODUCT`.

Puis création de `Revenue = Quantity × Price`.

*Pourquoi exclure les retours ?* Le but est de modéliser la **valeur générée par des achats effectifs** ; les annulations sont un autre phénomène (qualité, satisfaction) que l'on pourrait étudier séparément.

In [ ]:
df = df_raw.copy().rename(columns={"Customer ID": "CustomerID"})
df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

log = [("Dataset brut", len(df), 0)]
def step(name, mask_keep):
    global df
    before = len(df)
    df = df[mask_keep(df)]
    log.append((name, len(df), before - len(df)))

if DROP_DUPLICATES:
    step("1. Doublons exacts supprimés", lambda d: ~d.duplicated())
step("2. Annulations / retours (Invoice 'C...')", lambda d: ~d["Invoice"].str.startswith("C"))
step("3. Customer ID manquant", lambda d: d["CustomerID"].notna())
step("4. Quantity <= 0", lambda d: d["Quantity"] > 0)
step("5. Price <= 0", lambda d: d["Price"] > 0)
if REMOVE_NON_PRODUCT:
    step("6. Lignes non-produits (StockCode non standard)", lambda d: d["StockCode"].str.match(r"^\d{5}"))

df = df.copy()
df["CustomerID"] = df["CustomerID"].astype(int)
df["Revenue"] = df["Quantity"] * df["Price"]

clean_log = pd.DataFrame(log, columns=["Étape", "Lignes restantes", "Lignes supprimées"])
clean_log["% du brut supprimé"] = clean_log["Lignes supprimées"] / len(df_raw) * 100
display(clean_log)

comp = pd.DataFrame({
    "Avant": [len(df_raw), df_raw["Customer ID"].nunique(), df_raw["Invoice"].nunique(), df_raw["StockCode"].nunique()],
    "Après": [len(df), df["CustomerID"].nunique(), df["Invoice"].nunique(), df["StockCode"].nunique()],
}, index=["Lignes", "Clients", "Factures", "Produits (StockCode)"])
display(comp)
print(f"Revenue total après nettoyage : £{df['Revenue'].sum():,.0f}")
print(f"Période conservée : {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
display(df[["Quantity", "Price", "Revenue"]].describe(percentiles=[.01, .5, .99]))

**Interprétation :** le tableau montre l'impact de chaque règle. La suppression des `Customer ID` manquants est en général la plus lourde en nombre de lignes, mais pèse moins en revenu client-identifiable : ces transactions (invités / ventes non tracées) ne peuvent de toute façon pas alimenter un modèle *par client*. Après nettoyage, `Quantity`, `Price` et `Revenue` sont strictement positifs, ce qui rend les transformations logarithmiques possibles.

## 7. Exploratory Data Analysis

On analyse l'activité globale (volume, saisonnalité), le comportement des clients et la concentration de la valeur.

In [ ]:
orders = df.groupby("Invoice").agg(CustomerID=("CustomerID", "first"), Date=("InvoiceDate", "min"),
                                   Revenue=("Revenue", "sum"), Country=("Country", "first"))
cust_rev = df.groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False)
cust_orders = orders.groupby("CustomerID").size()

kpi = pd.Series({
    "Transactions (lignes)": len(df),
    "Factures": orders.shape[0],
    "Clients": df["CustomerID"].nunique(),
    "Produits (StockCode)": df["StockCode"].nunique(),
    "Pays": df["Country"].nunique(),
    "Chiffre d'affaires total (£)": df["Revenue"].sum(),
    "Panier moyen (£/facture)": orders["Revenue"].mean(),
    "Panier médian (£/facture)": orders["Revenue"].median(),
    "Commandes moyennes / client": cust_orders.mean(),
    "Commandes médianes / client": cust_orders.median(),
})
display(kpi.to_frame("Valeur"))
top10_share = cust_rev.head(int(0.1 * len(cust_rev))).sum() / cust_rev.sum() * 100
print(f"Les 10 % de clients les plus rentables génèrent {top10_share:.1f} % du chiffre d'affaires.")
print(f"Part des clients n'ayant passé qu'une seule commande : {(cust_orders == 1).mean() * 100:.1f} %")

In [ ]:
monthly = df.set_index("InvoiceDate").resample("MS").agg(Revenue=("Revenue", "sum"), Orders=("Invoice", "nunique"),
                                                         Customers=("CustomerID", "nunique"))
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(monthly.index, monthly["Revenue"], marker="o"); ax[0].set_title("Chiffre d'affaires mensuel (£)")
ax[0].tick_params(axis="x", rotation=45)
ax[1].bar(monthly.index, monthly["Orders"], width=20, color="tab:orange"); ax[1].set_title("Nombre de commandes par mois")
ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()
print("Mois au CA maximal :", monthly["Revenue"].idxmax().strftime("%Y-%m"), f"(£{monthly['Revenue'].max():,.0f})")
print("Mois au CA minimal :", monthly["Revenue"].idxmin().strftime("%Y-%m"), f"(£{monthly['Revenue'].min():,.0f})")

**Comment lire :** la courbe de CA et le nombre de commandes révèlent la **saisonnalité** (pic attendu en fin d'année pour un commerce de cadeaux). Conséquence directe pour la suite : la fenêtre « futur » dépendra de la période choisie, d'où l'indicateur `first_q4` (client acquis pendant le pic de Noël, souvent un acheteur ponctuel). Le dernier mois est généralement **incomplet** (la base s'arrête début décembre), ne pas l'interpréter comme un effondrement.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
sns.histplot(cust_rev, bins=60, log_scale=True, ax=ax[0]); ax[0].set_title("Distribution du revenu par client (échelle log)")
ax[0].set_xlabel("Revenu total du client (£)")
sns.histplot(cust_orders, bins=60, log_scale=True, ax=ax[1], color="tab:green")
ax[1].set_title("Distribution du nombre de commandes par client (échelle log)"); ax[1].set_xlabel("Nombre de commandes")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
cust_rev.head(10).sort_values().plot.barh(ax=ax[0], color="tab:purple"); ax[0].set_title("Top 10 clients (CA £)"); ax[0].set_ylabel("CustomerID")
top_prod = df.groupby("Description")["Quantity"].sum().nlargest(10).sort_values()
top_prod.plot.barh(ax=ax[1], color="tab:red"); ax[1].set_title("Top 10 produits (quantités vendues)"); ax[1].set_ylabel("")
cc = df.groupby("Country")["CustomerID"].nunique().sort_values(ascending=False).head(10).sort_values()
cc.plot.barh(ax=ax[2], color="tab:cyan"); ax[2].set_title("Clients par pays (Top 10)"); ax[2].set_ylabel("")
plt.tight_layout(); plt.show()

uk_share = df.loc[df["Country"] == "United Kingdom", "Revenue"].sum() / df["Revenue"].sum() * 100
print(f"Part du Royaume-Uni dans le CA : {uk_share:.1f} %")

**Interprétation :**
- *Distribution du revenu / des commandes* : fortement asymétrique (queue droite) → on utilisera des **transformations log** et des modèles robustes ; la moyenne est trompeuse, la médiane est plus représentative.
- *Top clients* : quelques clients pèsent beaucoup du CA (loi de Pareto) → leur perte coûte cher ; d'où l'intérêt d'un modèle CLV + churn.
- *Pays* : le Royaume-Uni domine fortement → `is_uk` est une variable pertinente, mais les autres pays sont très peu représentés (prudence dans l'interprétation par pays).

## 8. Customer-Level Dataset

On passe d'une table de transactions à **une ligne par client**. Pour éviter le data leakage plus tard, on écrit une **fonction unique** `build_features(data, snapshot)` qui ne regarde **que** les transactions antérieures à `snapshot` : on la réutilisera à plusieurs dates (cutoff de prédiction, landmark de survie, aujourd'hui).

Variables créées : `Recency`, `Frequency` (= `Total_Orders`), `Monetary`, `AOV`, `Unique_Products`, `First_Purchase`, `Last_Purchase`, `Lifetime`, `Country`, `is_uk`, `first_q4`, + `Tenure` (ancienneté), `avg_items_per_order`, `avg_days_between` (délai moyen entre commandes) et les versions **log**.

In [ ]:
END  = df["InvoiceDate"].max()
SNAP = END.normalize() + pd.Timedelta(days=1)      # "aujourd'hui" : lendemain du dernier jour de données

def build_features(data, snapshot):
    d = data[data["InvoiceDate"] < snapshot]
    inv = d.groupby(["CustomerID", "Invoice"]).agg(order_date=("InvoiceDate", "min"),
                                                  order_value=("Revenue", "sum"),
                                                  order_items=("Quantity", "sum")).reset_index()
    c = inv.groupby("CustomerID").agg(Frequency=("Invoice", "nunique"), Monetary=("order_value", "sum"),
                                      AOV=("order_value", "mean"), First_Purchase=("order_date", "min"),
                                      Last_Purchase=("order_date", "max"), avg_items_per_order=("order_items", "mean"))
    c["Total_Orders"] = c["Frequency"]
    c["Unique_Products"] = d.groupby("CustomerID")["StockCode"].nunique()
    c["Country"] = d.sort_values("InvoiceDate").groupby("CustomerID")["Country"].last()
    c["Recency"] = (snapshot - c["Last_Purchase"]).dt.days
    c["Lifetime"] = (c["Last_Purchase"] - c["First_Purchase"]).dt.days
    c["Tenure"] = (snapshot - c["First_Purchase"]).dt.days
    c["is_uk"] = (c["Country"] == "United Kingdom").astype(int)
    c["first_q4"] = c["First_Purchase"].dt.month.isin([10, 11, 12]).astype(int)
    # délai moyen entre commandes ; clients à 1 commande -> on met leur ancienneté (conservateur)
    c["avg_days_between"] = np.where(c["Frequency"] > 1, c["Lifetime"] / (c["Frequency"] - 1).clip(lower=1), c["Tenure"])
    # transformations log
    c["log_Monetary"] = np.log1p(c["Monetary"].clip(lower=0))
    c["log_AOV"] = np.log1p(c["AOV"].clip(lower=0))
    c["log_Frequency"] = np.log1p(c["Frequency"])
    c["log_Unique_Products"] = np.log1p(c["Unique_Products"])
    c["log_Lifetime"] = np.log1p(c["Lifetime"])
    c["log_items"] = np.log1p(c["avg_items_per_order"].clip(lower=0))
    c["Recency_30d"] = c["Recency"] / 30.0          # Recency en "mois" (coefficients Cox/Weibull lisibles)
    return c

cust = build_features(df, SNAP)
print("Table client :", cust.shape)
display(cust.head())
display(cust[["Recency", "Frequency", "Monetary", "AOV", "Unique_Products", "Lifetime", "Tenure"]].describe())

**Interprétation :** `Recency` faible = client récent ; `Frequency` et `Monetary` suivent des lois très asymétriques (moyenne ≫ médiane). Un `Lifetime` égal à 0 signifie un client ayant acheté **un seul jour** (très fréquent dans le e-commerce) : c'est la population la plus difficile à prédire.

## 9. RFM Analysis

**RFM** résume le comportement : *R*ecency (à quel point récent), *F*requency (à quel point fréquent), *M*onetary (à quel point dépensier). On note chaque dimension de 1 à 5 (quintiles ; 5 = meilleur). Pour la Recency, le score est **inversé** (peu de jours = bon score).

In [ ]:
rfm = cust[["Recency", "Frequency", "Monetary"]].copy()
rfm["R"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_Score"] = rfm["R"] + rfm["F"] + rfm["M"]

def rfm_profile(r):
    if r.R >= 4 and r.F >= 4: return "1. Très récents & très actifs"
    if r.F >= 4 and r.R >= 2: return "2. Fidèles"
    if r.M >= 4 and r.R <= 2: return "3. Gros dépensiers inactifs"
    if r.R >= 4:              return "4. Récents / nouveaux"
    if r.R <= 2 and r.F <= 2 and r.M <= 2: return "6. Faible valeur / perdus"
    return "5. Occasionnels"

rfm["RFM_Profile"] = rfm.apply(rfm_profile, axis=1)
prof = rfm.groupby("RFM_Profile").agg(Clients=("R", "size"), Recency_moy=("Recency", "mean"), Frequency_moy=("Frequency", "mean"),
                                      Monetary_moy=("Monetary", "mean"), Monetary_total=("Monetary", "sum"))
prof["% clients"] = prof["Clients"] / prof["Clients"].sum() * 100
prof["% CA"] = prof["Monetary_total"] / prof["Monetary_total"].sum() * 100
display(prof)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
prof["% clients"].plot.bar(ax=ax[0], color="tab:blue"); ax[0].set_title("% des clients par profil RFM")
prof["% CA"].plot.bar(ax=ax[1], color="tab:green"); ax[1].set_title("% du CA par profil RFM")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
cust["RFM_Profile"] = rfm["RFM_Profile"]; cust["RFM_Score"] = rfm["RFM_Score"]

**Signification business des profils :**

| Profil | Lecture | Action type |
|---|---|---|
| Très récents & très actifs | Le cœur du business | Fidéliser, récompenser |
| Fidèles | Achètent souvent, un peu moins récemment | Maintenir l'engagement |
| Gros dépensiers inactifs | **Ont beaucoup dépensé mais ne reviennent plus** : manque à gagner élevé | Réactivation prioritaire |
| Récents / nouveaux | Acheteurs récents peu fréquents | Onboarding, 2ᵉ achat |
| Occasionnels | Valeur moyenne, comportement irrégulier | Cross-sell léger |
| Faible valeur / perdus | Peu récents, peu fréquents, peu dépensiers | Automatisation à coût minimal |

**Limite de RFM :** c'est une photographie du **passé**. Il ne dit ni combien le client rapportera, ni quand il partira → d'où la suite du projet (régression + survie).

## 10. Feature Engineering

Variables retenues pour la prédiction. Quelques choix expliqués :

- **log(Monetary)**, **log(AOV)** : réduisent l'influence des valeurs extrêmes, utile surtout pour la régression linéaire ;
- **Recency, Frequency, Tenure, Lifetime** : dynamique d'achat ;
- **avg_days_between** : rythme d'achat du client ;
- **is_uk, first_q4** : contexte géographique et saisonnier d'acquisition ;
- `Total_Orders` (= `Frequency`) et `Monetary` brut sont **redondants** (Monetary = Frequency × AOV) : on garde les logs pour limiter la colinéarité.

In [ ]:
REV_FEATS = ["Recency", "Frequency", "log_Monetary", "log_AOV", "Unique_Products", "Lifetime", "Tenure",
             "avg_items_per_order", "avg_days_between", "is_uk", "first_q4"]

corr = cust[REV_FEATS].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=.5)
plt.title("Corrélations entre variables explicatives"); plt.tight_layout(); plt.show()

**Interprétation :** repérer les paires fortement corrélées (|ρ| > 0.8). Elles sont gênantes pour les modèles linéaires (coefficients instables) mais peu pour les arbres. C'est une raison de plus de comparer plusieurs familles de modèles.

## 11. Temporal Split (prévention du data leakage)

**Principe**
```
|---------- Observation period ----------|-------- Future period (180 j) --------|
début des données                      CUTOFF                                fin des données
 → calcul des variables (X)                       → revenu réel = cible (y)
```

**Pourquoi un split temporel et pas aléatoire ?**
Pour prédire la CLV on cherche à connaître le **futur** à partir du **passé**. Un split aléatoire de transactions mélangerait des achats postérieurs dans les variables historiques (ex. une Recency calculée *après* un achat qu'on cherche à prédire) : le modèle « tricherait » avec des informations indisponibles en production → **data leakage**, performances artificiellement excellentes et inutilisables. Ici :

- les variables X sont calculées **uniquement** avec les transactions `< CUTOFF` ;
- la cible `Future_Revenue` est le CA réalisé **dans les 180 jours suivant** `CUTOFF` (0 si le client n'a pas racheté) ;
- seuls les clients **connus avant** le cutoff sont prédits (un client qui apparaît après n'est pas prévisible).

Ensuite seulement, on sépare les **clients** en train (80 %) / test (20 %) pour évaluer le modèle sur des clients non vus.

In [ ]:
CUTOFF = (END - pd.Timedelta(days=HORIZON)).normalize()
obs = df[df["InvoiceDate"] < CUTOFF]
fut = df[df["InvoiceDate"] >= CUTOFF]
assert obs["InvoiceDate"].max() < CUTOFF <= fut["InvoiceDate"].min(), "Chevauchement temporel !"

print(f"Observation : {obs['InvoiceDate'].min().date()} → {obs['InvoiceDate'].max().date()}  ({obs['CustomerID'].nunique():,} clients)")
print(f"Futur       : {fut['InvoiceDate'].min().date()} → {fut['InvoiceDate'].max().date()}  ({fut['CustomerID'].nunique():,} clients actifs)")

feat_T = build_features(df, CUTOFF)                         # X : historique AVANT le cutoff uniquement
feat_T["Future_Revenue"] = fut.groupby("CustomerID")["Revenue"].sum().reindex(feat_T.index).fillna(0.0)   # y

print(f"\nClients modélisés : {len(feat_T):,}")
print(f"Part des clients avec Future_Revenue > 0 : {(feat_T['Future_Revenue'] > 0).mean() * 100:.1f} %")
display(feat_T["Future_Revenue"].describe(percentiles=[.5, .75, .9, .99]))

fig, ax = plt.subplots(figsize=(11, 2.2))
ax.barh(0, (CUTOFF - obs["InvoiceDate"].min()).days, left=0, color="tab:blue", label="Observation")
ax.barh(0, (END - CUTOFF).days, left=(CUTOFF - obs["InvoiceDate"].min()).days, color="tab:orange", label="Futur (cible)")
ax.set_yticks([]); ax.set_xlabel("Jours depuis le début des données"); ax.legend(loc="upper center", ncol=2)
plt.title("Découpage temporel"); plt.tight_layout(); plt.show()

**Interprétation :** la distribution de `Future_Revenue` est **« zero-inflated »** : une grande part de clients ne rachète pas pendant la fenêtre future (revenu = 0) et quelques clients concentrent l'essentiel du revenu. Cela explique que le R² soit difficile à obtenir et que MAE et RMSE puissent raconter des histoires différentes.

## 12. Future Revenue Prediction

**Objectif :** prédire `Future_Revenue` (CA sur les 180 prochains jours) à partir des variables de comportement.

**Pourquoi plusieurs modèles ?**
| Modèle | Rôle |
|---|---|
| Baseline (moyenne) | Référence minimale : tout modèle doit faire mieux |
| Linear Regression | Interprétable, capture les relations linéaires |
| Random Forest | Capture les non-linéarités / interactions, robuste |
| Gradient Boosting / XGBoost | Souvent le plus précis sur données tabulaires |
| GB sur cible log | Variante qui réduit l'effet des très gros clients |

Métriques : **MAE** (erreur moyenne en £), **RMSE** (pénalise fortement les grosses erreurs), **R²** (part de variance expliquée ; peut être faible/négatif).

In [ ]:
X_all, y_all = feat_T[REV_FEATS], feat_T["Future_Revenue"]
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=(y_all > 0))
print(f"Train : {len(X_tr):,} clients | Test : {len(X_te):,} clients")

models = {
    "Baseline (moyenne)": DummyRegressor(strategy="mean"),
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=RANDOM_STATE),
    "GB (cible log)": TransformedTargetRegressor(
        regressor=GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=RANDOM_STATE),
        func=np.log1p, inverse_func=np.expm1),
}
try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                                     colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1)
except Exception as e:
    print("XGBoost indisponible :", e)

rows, preds = [], {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = np.clip(m.predict(X_te), 0, None)       # un revenu ne peut pas être négatif
    preds[name] = p
    rows.append({"Model": name, "MAE": mean_absolute_error(y_te, p),
                 "RMSE": np.sqrt(mean_squared_error(y_te, p)), "R2": r2_score(y_te, p)})
res = pd.DataFrame(rows).set_index("Model")
base = res.loc["Baseline (moyenne)"]
res["Gain MAE vs baseline (%)"] = (1 - res["MAE"] / base["MAE"]) * 100
res["Gain RMSE vs baseline (%)"] = (1 - res["RMSE"] / base["RMSE"]) * 100
display(res.sort_values("RMSE"))

## 13. Model Comparison

On retient le modèle (hors baseline) au **RMSE le plus faible** sur le jeu de test, en vérifiant que MAE et R² vont dans le même sens. Le RMSE est choisi car, pour une CLV, **se tromper sur un gros client coûte cher**.

In [ ]:
BEST = res.drop(index="Baseline (moyenne)")["RMSE"].idxmin()
print(f"Meilleur modèle (RMSE test) : {BEST}")
best_model = models[BEST]
p_best = preds[BEST]

fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
res[["MAE", "RMSE"]].plot.bar(ax=ax[0]); ax[0].set_title("MAE / RMSE par modèle (test)"); ax[0].tick_params(axis="x", rotation=40)
ax[1].scatter(np.log1p(y_te), np.log1p(p_best), s=8, alpha=.35)
lim = max(np.log1p(y_te).max(), np.log1p(p_best).max()); ax[1].plot([0, lim], [0, lim], "r--")
ax[1].set_xlabel("log(1+ revenu réel)"); ax[1].set_ylabel("log(1+ revenu prédit)"); ax[1].set_title(f"Réel vs prédit — {BEST}")
resid = p_best - y_te.values
sns.histplot(np.clip(resid, np.percentile(resid, 1), np.percentile(resid, 99)), bins=60, ax=ax[2])
ax[2].set_title("Erreurs de prédiction (prédit − réel, écrêtées 1–99 %)")
plt.tight_layout(); plt.show()

# calibration par décile de prédiction
calib = pd.DataFrame({"pred": p_best, "reel": y_te.values})
calib["decile"] = pd.qcut(calib["pred"].rank(method="first"), 10, labels=False) + 1
cal_tab = calib.groupby("decile").agg(Prédit_moy=("pred", "mean"), Réel_moy=("reel", "mean"), Clients=("pred", "size"))
display(cal_tab)

# importance par permutation (valable pour tout type de modèle)
pi = permutation_importance(best_model, X_te, y_te, n_repeats=5, random_state=RANDOM_STATE,
                            scoring="neg_root_mean_squared_error", n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=REV_FEATS).sort_values()
imp.plot.barh(figsize=(8, 5), color="tab:green"); plt.title(f"Feature importance (permutation, hausse de RMSE) — {BEST}")
plt.tight_layout(); plt.show()
display(imp.sort_values(ascending=False).to_frame("Importance"))

**Explicabilité — régression**

- **Pourquoi ce modèle ?** Le modèle retenu est celui qui minimise l'erreur sur des clients **jamais vus** ; les arbres/boosting gèrent les non-linéarités et les effets de seuil typiques du comportement d'achat.
- **Variables importantes ?** Lire le graphique de permutation : plus l'erreur augmente quand on mélange une variable, plus le modèle en dépend (en général : intensité d'achat passée — `log_Monetary`, `Frequency`, `AOV` — et `Recency`).
- **Résultat ?** Comparer MAE/RMSE/R² au baseline (colonnes « Gain vs baseline »). Un gain positif prouve que l'historique contient de l'information prédictive. Un R² modeste est **normal** : le futur d'un client est intrinsèquement incertain.
- **Lire la calibration :** si les déciles de prédiction croissent comme le réel moyen, le modèle **classe** bien les clients — ce qui suffit pour prioriser des actions marketing, même si le niveau exact est approximatif.
- **Décision business :** utiliser les prédictions pour cibler les clients au revenu futur attendu élevé (budget marketing proportionnel à la valeur attendue).

In [ ]:
# Modèle final de revenu : ré-entraîné sur tous les clients du cutoff (utilisé plus tard sur les features "aujourd'hui")
final_rev_model = clone(best_model).fit(X_all, y_all)
print("Modèle final de revenu entraîné sur", len(X_all), "clients")

## 14. Survival Analysis

### Pourquoi la Survival Analysis ?
Un classifieur « churn oui/non » ignore **quand** le client part et traite mal les clients encore actifs. La Survival Analysis modélise le **temps jusqu'à l'événement** (churn) en exploitant l'information partielle des clients qui n'ont pas (encore) churné.

### Définition du churn
Un client **churne** s'il passe **plus de `thr` jours sans acheter** (thr = 180 retenu ; 90 et 270 en sensibilité).

### Censure à droite (right-censoring)
À la fin des données (9 déc. 2011), un client qui a acheté récemment n'a **pas** churné… *mais on ne sait pas s'il churnera plus tard*. On connaît seulement qu'il est resté actif **au moins** jusqu'à la fin de l'observation : sa vraie durée est **≥** la durée observée. Ce n'est pas une donnée manquante, c'est une **information partielle** :
- `Event = 1` : churn observé ;
- `Event = 0` : client encore actif → **censuré à droite** (on l'exclure ou le traiter comme « jamais parti » biaiserait la durée de vie vers le bas ou le haut).

### Construction (design « landmark », sans leakage)
On se place à une date **landmark** = `fin des données − 365 jours`. 
1. Population : clients **actifs** au landmark (Recency ≤ `thr`) — on ne peut pas « faire churner » deux fois un client déjà parti.
2. Variables (`Recency`, `Frequency`, …) : calculées **avec les seules transactions avant le landmark**.
3. Temps `Duration` : jours écoulés depuis le landmark jusqu'à la date de churn (`dernier achat + thr`) si le churn est observé ; sinon `365` (censure à la fin des données).

> *Pourquoi ne pas utiliser `Recency` et `Lifetime` calculés à la fin des données ?* Parce qu'ils **définissent** mécaniquement l'événement et la durée : le modèle redécouvrirait la définition au lieu d'apprendre un comportement (leakage). Avec le landmark, `Recency` est une vraie variable prédictive.

In [ ]:
LANDMARK = SNAP - pd.Timedelta(days=SURV_FOLLOWUP)
last_all = df.groupby("CustomerID")["InvoiceDate"].max()

def make_survival_table(thr):
    f = build_features(df, LANDMARK)
    f = f[f["Recency"] <= thr].copy()                     # clients actifs au landmark
    la = last_all.reindex(f.index)
    f["Event"] = ((SNAP - la).dt.days > thr).astype(int)      # churn observé avant la fin des données ?
    churn_time = (la - LANDMARK).dt.days + thr
    f["Duration"] = np.where(f["Event"] == 1, churn_time, (SNAP - LANDMARK).days).astype(float)
    f["Duration"] = f["Duration"].clip(lower=1)
    return f

print("Landmark :", LANDMARK.date(), "| Fin des données :", END.date())
surv_tabs = {thr: make_survival_table(thr) for thr in THRESHOLDS}
for thr, t in surv_tabs.items():
    print(f"thr={thr:>3} j : {len(t):,} clients actifs au landmark | churn observé : {t['Event'].mean()*100:5.1f} % | censurés : {(1-t['Event'].mean())*100:5.1f} %")
display(surv_tabs[CHURN_DAYS][["Recency", "Frequency", "Monetary", "Duration", "Event"]].head())

### Comparaison des seuils 90 / 180 / 270 jours
On compare, pour chaque seuil : taille de la population, taux de churn, durée médiane et survie à 180 jours (Kaplan-Meier).

In [ ]:
rows = []
fig, ax = plt.subplots(figsize=(9, 5))
for thr, t in surv_tabs.items():
    k = KaplanMeierFitter().fit(t["Duration"], t["Event"], label=f"thr = {thr} j")
    k.plot_survival_function(ax=ax, ci_show=False)
    rows.append({"Seuil (jours)": thr, "Clients actifs au landmark": len(t), "Taux de churn observé (%)": t["Event"].mean() * 100,
                 "Durée médiane de survie (j)": k.median_survival_time_,
                 "S(180 j)": float(k.predict(180)), "S(365 j)": float(k.predict(SURV_FOLLOWUP))})
ax.set_xlabel("Jours depuis le landmark"); ax.set_ylabel("Probabilité de rester actif"); ax.set_title("Kaplan-Meier selon le seuil de churn")
plt.tight_layout(); plt.show()
display(pd.DataFrame(rows).set_index("Seuil (jours)"))

**Pourquoi retenir 180 jours ?** (à confirmer avec le tableau ci-dessus)
- **90 j** : trop sensible — un client saisonnier ou B2B qui commande tous les 3-4 mois serait déclaré « parti » à tort (faux churn) ;
- **270 j** : trop tardif — on détecte le départ quand il est trop tard pour agir, et on perd des événements observables dans la fenêtre (plus de censure) ;
- **180 j** : compromis (≈ 6 mois = un cycle d'achat complet y compris saison haute), horizon cohérent avec la fenêtre de prédiction du revenu (`HORIZON = 180`), assez long pour limiter les faux churns et assez court pour déclencher une action marketing utile.

## 15. Kaplan-Meier

L'estimateur de **Kaplan-Meier** donne `S(t) = P(T > t)` : la probabilité qu'un client **actif au landmark** n'ait pas encore churné au bout de `t` jours. Il gère correctement la censure. On compare ensuite des groupes (test du log-rank : H0 = « mêmes courbes »).

In [ ]:
surv = surv_tabs[CHURN_DAYS].copy()
kmf = KaplanMeierFitter().fit(surv["Duration"], surv["Event"], label="Tous les clients actifs")
print("Durée médiane de survie :", kmf.median_survival_time_, "jours (inf = la survie ne passe pas sous 50 % sur l'horizon observé)")
for t in [30, 90, 180, 270, 365]:
    print(f"S({t:>3} j) = {float(kmf.predict(t)):.3f}")

surv["Freq_group"] = pd.cut(surv["Frequency"], [0, 1, 4, np.inf], labels=["1 commande", "2-4 commandes", "5+ commandes"])
groups = {"Pays": ("is_uk", {0: "Hors UK", 1: "UK"}), "Acquisition": ("first_q4", {0: "Hors Q4", 1: "Acquis en Q4"}),
          "Fréquence": ("Freq_group", None)}
fig, ax = plt.subplots(1, 4, figsize=(22, 4.8))
kmf.plot_survival_function(ax=ax[0]); ax[0].set_title("Kaplan-Meier global")
for a, (title, (col, mp)) in zip(ax[1:], groups.items()):
    for val, g in surv.groupby(col):
        if len(g) < 20: continue
        KaplanMeierFitter().fit(g["Duration"], g["Event"], label=str(mp[val] if mp else val)).plot_survival_function(ax=a, ci_show=False)
    a.set_title(f"KM par {title}")
for a in ax: a.set_xlabel("Jours depuis le landmark"); a.set_ylabel("P(rester actif)")
plt.tight_layout(); plt.show()

for title, (col, mp) in groups.items():
    r = multivariate_logrank_test(surv["Duration"], surv[col].astype(str), surv["Event"])
    print(f"Log-rank ({title}) : p-value = {r.p_value:.3g}")

**Interprétation business :**
- La courbe globale descend plus ou moins vite selon la fidélité de la base : une chute rapide au début signale que **les premiers mois sont critiques** (période où agir pour fidéliser).
- Si la médiane est `inf`, plus de 50 % des clients actifs sont encore actifs à la fin de la fenêtre de suivi.
- Des courbes **séparées** et une p-value du log-rank faible signifient que le groupe compte réellement pour la rétention. En général, plus un client a de commandes, plus sa courbe est haute (les fidèles restent) → **investir dans le 2ᵉ et 3ᵉ achat**.

## 16. Cox Proportional Hazards

Le modèle de **Cox** relie le risque instantané de churn aux variables : `h(t|x) = h0(t)·exp(β·x)`.
- **β > 0 / Hazard Ratio (exp β) > 1** : la variable **augmente** le risque de churn ;
- **β < 0 / HR < 1** : la variable **diminue** le risque.

**Variables :** `Recency_30d` (en mois), `log_Frequency`, `log_AOV`, `log_Unique_Products`, `log_Lifetime`, `log_items`, `is_uk`, `first_q4`.
`Total_Orders` (= Frequency) et `Monetary` (= Frequency × AOV) sont **volontairement exclus** : colinéarité parfaite/forte qui rendrait les coefficients non interprétables.
Évaluation : **C-index** sur clients de test (0.5 = hasard, > 0.60 = objectif indicatif).

In [ ]:
SURV_FEATS = ["Recency_30d", "log_Frequency", "log_AOV", "log_Unique_Products", "log_Lifetime", "log_items", "is_uk", "first_q4"]
cols = SURV_FEATS + ["Duration", "Event"]
s_tr, s_te = train_test_split(surv[cols], test_size=0.2, random_state=RANDOM_STATE, stratify=surv["Event"])

cph = CoxPHFitter(penalizer=0.01).fit(s_tr, duration_col="Duration", event_col="Event")
cindex_cox = cph.score(s_te, scoring_method="concordance_index")
print(f"C-index Cox — train : {cph.concordance_index_:.3f} | test : {cindex_cox:.3f}")
sm = cph.summary[["coef", "exp(coef)", "p"]].rename(columns={"exp(coef)": "Hazard Ratio"})
sm["Effet sur le churn"] = np.where(sm["coef"] > 0, "↑ augmente le risque", "↓ diminue le risque")
display(sm.sort_values("Hazard Ratio", ascending=False))

plt.figure(figsize=(8, 4.5)); cph.plot(hazard_ratios=True); plt.title("Cox — Hazard ratios (IC 95 %)")
plt.tight_layout(); plt.show()

print("\nVérification de l'hypothèse des risques proportionnels (test de Schoenfeld) :")
try:
    cph.check_assumptions(s_tr, p_value_threshold=0.01, show_plots=False)
except Exception as e:
    print("check_assumptions indisponible :", e)

**Explicabilité — Cox**
- **Pourquoi ce modèle ?** Il est semi-paramétrique, interprétable (hazard ratios) et gère la censure ; idéal pour *comprendre* les facteurs de churn.
- **Variables importantes ?** Lire les HR significatifs (p < 0.05). Typiquement : une `Recency` élevée **augmente** le risque ; la fréquence, la diversité de produits ou un historique long **diminuent** le risque.
- **Résultat ?** Comparer le C-index test à 0.60 (objectif indicatif) : il mesure la capacité à **ordonner** les clients du plus au moins risqué.
- **Hypothèse PH :** si Schoenfeld rejette pour certaines variables (fréquent sur de gros échantillons), l'effet varie avec le temps : le HR est alors un **effet moyen**. On peut stratifier ou utiliser le Weibull (section suivante) comme alternative paramétrique ; l'interprétation reste exploitable mais à nuancer.
- **Décision business :** surveiller en priorité les facteurs qui augmentent le hazard (ex. inactivité qui s'allonge) et déclencher les relances **avant** le seuil de 180 jours.

## 17. Weibull Survival Model

Le **Weibull AFT** (Accelerated Failure Time) est un modèle **paramétrique** : `S(t|x) = exp(-(t/λ(x))^ρ)` avec `λ(x) = exp(β·x)`.
Contrairement à Cox, il fournit une **fonction de survie complète** pour tout `t`, donc :
- la **probabilité de rester actif** `P(T > t)` à n'importe quel horizon ;
- la **durée attendue / médiane** ;
- la survie **future** nécessaire pour la CLV (à 180, 360, 540 jours…).

`ρ` (forme) : `ρ > 1` risque de churn qui augmente avec le temps ; `ρ < 1` risque qui diminue (les clients qui survivent aux premiers mois deviennent plus fidèles).

In [ ]:
waft = WeibullAFTFitter(penalizer=0.01).fit(s_tr, duration_col="Duration", event_col="Event")
cindex_w = waft.score(s_te, scoring_method="concordance_index")
print(f"C-index Weibull AFT — test : {cindex_w:.3f}")
display(waft.summary[["coef", "exp(coef)", "p"]])
rho = float(np.exp(waft.params_.loc[("rho_", "Intercept")]))
print(f"\nParamètre de forme ρ = {rho:.3f} →", "le risque de churn augmente avec le temps" if rho > 1 else "le risque de churn diminue avec le temps")

def weibull_S(model, data, t):
    # S(t|x) = exp(-(t/lambda(x))^rho) ; t scalaire ou tableau
    p = model.params_
    lam = np.exp(data[SURV_FEATS].values @ p["lambda_"][SURV_FEATS].values + p["lambda_"]["Intercept"])
    r = np.exp(p["rho_"]["Intercept"])
    return np.exp(-(np.asarray(t, dtype=float) / lam) ** r)

# contrôle de cohérence avec lifelines
chk = waft.predict_survival_function(s_te[SURV_FEATS].head(3), times=[180]).T.values.ravel()
print("Contrôle S(180) lifelines vs formule :", np.round(chk, 4), np.round(weibull_S(waft, s_te.head(3), 180), 4))

# durée médiane attendue et validation du risque à 180 j (AUC)
s_te_ = s_te.copy()
s_te_["Churn_Risk_180"] = 1 - weibull_S(waft, s_te_, 180)
y_true = ((s_te_["Event"] == 1) & (s_te_["Duration"] <= 180)).astype(int)    # churn réalisé dans les 180 jours
auc = roc_auc_score(y_true, s_te_["Churn_Risk_180"])
print(f"AUC du Churn_Risk à 180 j (churn réalisé dans les 180 j, test) : {auc:.3f}")
print("Durée médiane prédite (clients test) — médiane des médianes :", float(waft.predict_median(s_te_[SURV_FEATS]).median()), "jours")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
tt = np.linspace(1, 365 * 2, 200)
for q, lab in [(0.1, "10 % — profil à risque"), (0.5, "médian"), (0.9, "90 % — profil fidèle")]:
    row = s_te_[SURV_FEATS].quantile(q).to_frame().T if False else s_te_.sort_values("Churn_Risk_180").iloc[[int((1 - q) * (len(s_te_) - 1))]]
    ax[0].plot(tt, [float(weibull_S(waft, row, t)[0]) for t in tt], label=lab)
ax[0].set_title("Weibull : courbes de survie de 3 profils (jusqu'à 2 ans)"); ax[0].set_xlabel("Jours"); ax[0].set_ylabel("P(T > t)"); ax[0].legend()
sns.histplot(s_te_["Churn_Risk_180"], bins=40, ax=ax[1]); ax[1].set_title("Distribution du risque de churn à 180 j (test)")
plt.tight_layout(); plt.show()

# Modèles finaux (tous les clients du cohort) pour l'usage en production
waft_final = WeibullAFTFitter(penalizer=0.01).fit(surv[cols], duration_col="Duration", event_col="Event")
cph_final = CoxPHFitter(penalizer=0.01).fit(surv[cols], duration_col="Duration", event_col="Event")
print("Modèles de survie finaux entraînés sur", len(surv), "clients")

**Explicabilité — Weibull**
- **Pourquoi ?** Il donne `P(T > t)` pour **n'importe quel horizon**, indispensable pour projeter la valeur sur plusieurs périodes.
- **Résultat ?** Le C-index (ordre des clients) et l'AUC du risque à 180 j mesurent la qualité ; comparer au Cox.
- **Interprétation :** `exp(coef) > 1` allonge la durée de vie (client plus durable) ; `< 1` la raccourcit. 
- **Décision :** `Churn_Risk = 1 − S(180 j | x)` classe les clients ; `S(k·180 | x)` alimente la CLV.
- **Limite :** la survie au-delà de 365 j est une **extrapolation** (le suivi observé est de 365 j) ; l'hypothèse Weibull doit être considérée comme une approximation.

## 18. CLV Calculation

### Approche retenue (prédictive, robuste)

On évalue chaque client **aujourd'hui** (`SNAP` = lendemain du dernier jour de données) avec les mêmes fonctions de features que pendant l'entraînement :

1. `Future_Revenue` = revenu prédit sur les 180 prochains jours (modèle de régression de la section 13) ;
2. `Survival_Probability` = `S(180 j | x)` du Weibull (0 si le client a **déjà churné**, c.-à-d. Recency > 180 j) ;
3. `Churn_Risk = 1 − Survival_Probability` ;
4. **CLV prédite sur K périodes de 180 jours**, actualisée :

```
Predicted_CLV = Marge × Revenu_prédit_180j × Σ_{k=0}^{K-1}  S(k·180 j | x) / (1 + r)^{k·180/365}
```
avec `S(0) = 1`, K = 6 (3 ans), marge = 30 %, r = 10 %/an.

*Intuition :* la première période vaut le revenu prédit ; chaque période suivante n'est comptée que si le client est encore actif, pondérée par sa probabilité de survie.

> **Hypothèses/limites :** revenu supposé stable d'une période à l'autre ; survie au-delà de 365 j extrapolée ; la marge (30 %) est constante.

### CLV de référence (formule classique)
`CLV_formule = AOV × Fréquence_annuelle × Durée_de_vie_attendue(années) × Marge`, avec une durée de vie attendue = `1 / taux de churn annuel` issu de Kaplan-Meier (plafonnée à 5 ans). Sert de **comparaison**.

In [ ]:
feat_end = build_features(df, SNAP)                              # features "aujourd'hui"
R_hat = np.clip(final_rev_model.predict(feat_end[REV_FEATS]), 0, None)
active = (feat_end["Recency"] <= CHURN_DAYS).values

disc = (1 + DISCOUNT_RATE) ** (HORIZON / 365)
Sk = [np.ones(len(feat_end))] + [weibull_S(waft_final, feat_end, k * HORIZON) * active for k in range(1, CLV_PERIODS)]
clv_mult = sum(Sk[k] / disc ** k for k in range(CLV_PERIODS))

out = pd.DataFrame(index=feat_end.index)
out["Country"] = feat_end["Country"]; out["is_uk"] = feat_end["is_uk"]
out["Recency"] = feat_end["Recency"]; out["Frequency"] = feat_end["Frequency"]
out["Monetary"] = feat_end["Monetary"]; out["AOV"] = feat_end["AOV"]
out["Lifetime"] = feat_end["Lifetime"]; out["Unique_Products"] = feat_end["Unique_Products"]
out["Already_Churned"] = (~active).astype(int)
out["Future_Revenue"] = R_hat                                         # prédit (180 j)
out["Survival_Probability"] = weibull_S(waft_final, feat_end, HORIZON) * active
out["Churn_Risk"] = 1 - out["Survival_Probability"]
out["Predicted_CLV"] = MARGIN * R_hat * clv_mult                       # valeur ESPÉRÉE (pondérée par la survie)
out["CLV_if_retained"] = MARGIN * R_hat * sum(1 / disc ** k for k in range(CLV_PERIODS))   # valeur SI le client est conservé (sans pondération par la survie)
out["Historical_CLV"] = MARGIN * feat_end["Monetary"]                  # profit déjà réalisé (30 % du CA)

# CLV "formule" de référence
s_km = KaplanMeierFitter().fit(surv["Duration"], surv["Event"])
annual_churn = max(1 - float(s_km.predict(SURV_FOLLOWUP)), 0.05)
life_years = min(5.0, 1 / annual_churn)
freq_year = feat_end["Frequency"] / feat_end["Tenure"].clip(lower=30) * 365
out["CLV_Formula"] = feat_end["AOV"] * freq_year * life_years * MARGIN
print(f"Churn annuel (KM) = {annual_churn:.2%} → durée de vie attendue = {life_years:.2f} ans")
display(out[["Historical_CLV", "Predicted_CLV", "CLV_Formula", "Future_Revenue", "Churn_Risk"]].describe())

In [ ]:
# ---- Comparaison de la CLV historique / prédite / réelle sur une période rétro-testée (cutoff) ----
idx = X_te.index
bt = pd.DataFrame({"Historique (30 % × CA passé)": MARGIN * feat_T.loc[idx, "Monetary"].values,
                   "Prédit (30 % × revenu prédit 180j)": MARGIN * p_best,
                   "Réel (30 % × revenu futur réel)": MARGIN * y_te.values}, index=idx)
def capture(score, actual, q=0.10):
    k = max(1, int(len(score) * q)); top = np.argsort(-np.asarray(score))[:k]
    return np.asarray(actual)[top].sum() / np.sum(actual) * 100
rows = []
for c in bt.columns[:2]:
    rows.append({"Score": c, "Spearman vs réel": spearmanr(bt[c], bt.iloc[:, 2])[0],
                 "% du revenu futur réel capté par le top 10 %": capture(bt[c], bt.iloc[:, 2]),
                 "MAE vs profit réel (£)": mean_absolute_error(bt.iloc[:, 2], bt[c])})
display(pd.DataFrame(rows).set_index("Score"))
print("Total profit futur réel (test) :", f"£{bt.iloc[:, 2].sum():,.0f}", "| total prédit :", f"£{bt.iloc[:, 1].sum():,.0f}",
      "| total historique :", f"£{bt.iloc[:, 0].sum():,.0f}")

**Interprétation :** on compare **ce qu'un score historique aurait fait** (CLV historique = « les gros clients d'hier sont les gros clients de demain ») à **ce que prédit le modèle**. Si le modèle capte une plus grande part du revenu futur réel dans son top 10 %, et classe mieux (Spearman plus élevé), la CLV prédictive apporte une vraie valeur par rapport à une simple lecture du passé. Note : le revenu futur réel n'est connu que pour le rétro-test ; la CLV « aujourd'hui » ne peut pas encore être vérifiée.

## 19. Customer Segmentation

Deux dimensions :
- **Valeur** : `High CLV` si `CLV_if_retained` (valeur du client **s'il est conservé**, sans pondération par la survie) ≥ quantile `CLV_HIGH_Q` (top 25 %), sinon Low/Medium. *On n'utilise pas `Predicted_CLV` ici car elle est déjà multipliée par la probabilité de survie : un client à haut risque aurait mécaniquement une CLV basse et le segment At Risk serait vide.* `Predicted_CLV` (valeur espérée) reste utilisée pour le Priority_Score et les KPI ;
- **Risque** : `High Risk` si `Churn_Risk` ≥ `RISK_THRESHOLD` (50 %).

| | Low Risk | High Risk |
|---|---|---|
| **High CLV** (valeur si conservé) | **Champions** | **At Risk** (prioritaire) |
| **Low/Medium CLV** | **Potential Loyalists** | **Low Value** |

Les clients déjà churnés (`Recency > 180 j`) ont `Churn_Risk = 1` → ils tombent côté *High Risk*.

In [ ]:
# Dimension "valeur" = valeur SI le client est conservé (CLV_if_retained), indépendante de la probabilité de survie.
# (Predicted_CLV est déjà pondérée par la survie : l'utiliser ici rendrait High CLV et High Risk quasi incompatibles
#  et viderait le segment At Risk.)
clv_thr = out["CLV_if_retained"].quantile(CLV_HIGH_Q)
out["Customer_Value"] = np.where(out["CLV_if_retained"] >= clv_thr, "High CLV", "Low CLV")
out["Risk_Level"] = np.where(out["Churn_Risk"] >= RISK_THRESHOLD, "High Risk", "Low Risk")
out["CLV_Level"] = pd.cut(out["CLV_if_retained"], [-np.inf, out["CLV_if_retained"].quantile(.5), clv_thr, np.inf], labels=["Low", "Medium", "High"])

def seg(r):
    if r["Customer_Value"] == "High CLV" and r["Risk_Level"] == "Low Risk":  return "Champions"
    if r["Customer_Value"] == "High CLV" and r["Risk_Level"] == "High Risk": return "At Risk"
    if r["Customer_Value"] == "Low CLV"  and r["Risk_Level"] == "Low Risk":  return "Potential Loyalists"
    return "Low Value"
out["Segment"] = out.apply(seg, axis=1)

# Score de priorité marketing : valeur espérée menacée = CLV × risque, sur les clients encore récupérables (non churnés)
out["CLV_at_Risk"] = out["Predicted_CLV"] * out["Churn_Risk"]
mask = out["Already_Churned"] == 0
out["Priority_Score"] = 0.0
out.loc[mask, "Priority_Score"] = out.loc[mask, "CLV_at_Risk"].rank(pct=True) * 100       # 0–100

SEG_ORDER = ["Champions", "Potential Loyalists", "At Risk", "Low Value"]
SEG_COL = {"Champions": "#2ca02c", "Potential Loyalists": "#1f77b4", "At Risk": "#d62728", "Low Value": "#7f7f7f"}
segsum = out.groupby("Segment").agg(Clients=("Segment", "size"), CLV_moyenne=("Predicted_CLV", "mean"), CLV_totale=("Predicted_CLV", "sum"),
                                    Churn_Risk_moyen=("Churn_Risk", "mean"), Revenu_futur_prédit=("Future_Revenue", "sum"),
                                    Recency_moy=("Recency", "mean"), Frequency_moy=("Frequency", "mean"),
                                    Monetary_moy=("Monetary", "mean")).reindex(SEG_ORDER).fillna(0)
segsum["% clients"] = segsum["Clients"] / segsum["Clients"].sum() * 100
segsum["% CLV totale"] = segsum["CLV_totale"] / segsum["CLV_totale"].sum() * 100
segsum = segsum.fillna(0)
empty = segsum.index[segsum["Clients"] == 0].tolist()
if empty: print("⚠️ Segments vides :", empty, "→ ajuste CLV_HIGH_Q / RISK_THRESHOLD")
display(segsum)
print(f"Seuil High CLV (quantile {CLV_HIGH_Q:.0%} de CLV_if_retained) : £{clv_thr:,.0f}")

# Évaluation de la segmentation : différences significatives entre segments ?
from scipy.stats import kruskal
for var in ["Predicted_CLV", "Churn_Risk"]:
    grp = [out.loc[out["Segment"] == s, var] for s in SEG_ORDER if (out["Segment"] == s).sum() > 1]
    if len(grp) >= 2:
        st, p = kruskal(*grp)
        print(f"Kruskal-Wallis {var} entre segments : p = {p:.3g}")

**Évaluation de la segmentation**
- **Cohérence des groupes :** `Champions` doivent avoir CLV élevée & risque faible ; `At Risk` CLV élevée & risque élevé, etc. (tableau ci-dessus — c'est vrai *par construction* sur les 2 axes, mais le tableau montre aussi `Recency`, `Frequency`, `Monetary` qui ne sont pas utilisés directement pour couper : s'ils sont cohérents, la segmentation est crédible).
- **Différences entre segments :** les tests de Kruskal-Wallis vérifient que CLV et risque diffèrent réellement.
- **Pertinence business :** regarder `% clients` vs `% CLV totale` : une petite fraction de clients concentre généralement l'essentiel de la valeur (Pareto) → justifie un traitement différencié.
- **Sensibilité :** les seuils (`CLV_HIGH_Q`, `RISK_THRESHOLD`) sont des paramètres métier modifiables (haut du notebook).

In [ ]:
print("TOP 20 — clients à contacter en priorité (valeur menacée la plus élevée parmi les clients encore actifs)\n")
top20 = (out[mask].sort_values("CLV_at_Risk", ascending=False).head(20)
         [["Segment", "Predicted_CLV", "Churn_Risk", "Survival_Probability", "Future_Revenue", "Recency", "Frequency", "Monetary", "Country", "Priority_Score"]])
display(top20)

## 20. Marketing Recommendations

| Segment | Profil | Actions recommandées | Budget / intensité |
|---|---|---|---|
| **Champions** (High CLV, Low Risk) | Très précieux et fidèles | Programme VIP, accès anticipé, offres personnalisées, récompenses, parrainage | Moyen (préserver, ne pas sur-remiser) |
| **Potential Loyalists** (Low/Med CLV, Low Risk) | Actifs avec potentiel | Cross-selling, upselling, recommandations personnalisées, incitation à la 2ᵉ/3ᵉ commande | Moyen-faible |
| **At Risk** (High CLV, High Risk) ⚠️ **Priorité n°1** | Précieux mais sur le point de partir | Contact prioritaire (téléphone / account manager pour B2B), campagne de réactivation, remise ciblée, enquête de satisfaction | **Élevé** : le ROI potentiel est maximal |
| **Low Value** (Low CLV, High Risk) | Faible valeur, risque élevé | Emails automatisés peu coûteux, newsletters, pas d'effort manuel | Minimal |

**Règle de pilotage :** budget de rétention ∝ `CLV_at_Risk` (= CLV × risque). Dépenser plus pour un client qu'il ne rapportera est une erreur : le coût d'une action doit rester **inférieur à la marge attendue** (`Predicted_CLV`).

In [ ]:
actions = pd.DataFrame({
    "Segment": SEG_ORDER,
    "Action": ["Fidélisation VIP / offres personnalisées", "Cross-sell, up-sell, 2ᵉ achat", "Réactivation prioritaire, contact humain, offre ciblée", "Automatisation low-cost"],
    "Priorité": ["2 — Maintenir", "3 — Développer", "1 — Urgent", "4 — Minimal"]}).set_index("Segment")
display(segsum[["Clients", "CLV_moyenne", "Churn_Risk_moyen"]].join(actions))

## 21. Visualizations

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(20, 11))
sns.histplot(out["Predicted_CLV"].clip(upper=out["Predicted_CLV"].quantile(.99)), bins=60, ax=ax[0, 0], color="tab:blue")
ax[0, 0].set_title("1. Distribution de la CLV prédite (écrêtée au 99ᵉ pct)"); ax[0, 0].set_xlabel("Predicted_CLV (£)")
sns.histplot(out["Churn_Risk"], bins=40, ax=ax[0, 1], color="tab:red"); ax[0, 1].set_title("2. Distribution du risque de churn")
sns.boxplot(data=out, x="Segment", y="Predicted_CLV", order=SEG_ORDER, palette=SEG_COL, showfliers=False, ax=ax[0, 2])
ax[0, 2].set_title("3. CLV par segment"); ax[0, 2].tick_params(axis="x", rotation=20)
sns.boxplot(data=out, x="Segment", y="Churn_Risk", order=SEG_ORDER, palette=SEG_COL, ax=ax[1, 0])
ax[1, 0].set_title("4. Churn risk par segment"); ax[1, 0].tick_params(axis="x", rotation=20)
cnt = out["Segment"].value_counts().reindex(SEG_ORDER)
ax[1, 1].bar(cnt.index, cnt.values, color=[SEG_COL[s] for s in cnt.index]); ax[1, 1].set_title("5. Nombre de clients par segment")
ax[1, 1].tick_params(axis="x", rotation=20)
for i, v in enumerate(cnt.values): ax[1, 1].text(i, v, f"{v:,}", ha="center", va="bottom")
top_clv = out.sort_values("Predicted_CLV", ascending=False).head(10).iloc[::-1]
ax[1, 2].barh(top_clv.index.astype(str), top_clv["Predicted_CLV"], color=[SEG_COL[s] for s in top_clv["Segment"]])
ax[1, 2].set_title("10. Top 10 clients selon la CLV prédite"); ax[1, 2].set_xlabel("£"); ax[1, 2].set_ylabel("CustomerID")
plt.tight_layout(); plt.show()

In [ ]:
# 6. CLV vs churn risk (scatter par segment)
plt.figure(figsize=(10, 6.5))
for s in SEG_ORDER:
    d = out[out["Segment"] == s]
    plt.scatter(d["Churn_Risk"], d["Predicted_CLV"], s=14, alpha=.5, label=f"{s} ({len(d):,})", color=SEG_COL[s])
plt.yscale("symlog", linthresh=10)
plt.axvline(RISK_THRESHOLD, color="k", ls="--", lw=1); plt.axhline(clv_thr, color="k", ls="--", lw=1)
plt.xlabel("Churn Risk"); plt.ylabel("Predicted CLV (£, échelle symlog)"); plt.title("6. CLV prédite vs risque de churn, par segment")
plt.legend(); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(20, 5))
# 7. Kaplan-Meier (global + par fréquence)
kmf.plot_survival_function(ax=ax[0]); ax[0].set_title("7. Courbe de survie Kaplan-Meier"); ax[0].set_xlabel("Jours"); ax[0].set_ylabel("P(rester actif)")
# 8. Revenu réel vs prédit (test, rétro-test)
ax[1].scatter(y_te, p_best, s=8, alpha=.35); mx = max(y_te.max(), p_best.max()); ax[1].plot([0, mx], [0, mx], "r--")
ax[1].set_xscale("symlog", linthresh=10); ax[1].set_yscale("symlog", linthresh=10)
ax[1].set_xlabel("Revenu futur réel (£)"); ax[1].set_ylabel("Revenu prédit (£)"); ax[1].set_title(f"8. Revenu réel vs prédit — {BEST}")
# 9. Feature importance
imp.plot.barh(ax=ax[2], color="tab:green"); ax[2].set_title("9. Feature importance (permutation)")
plt.tight_layout(); plt.show()

## 22. Business Insights

Les phrases ci-dessous sont **générées à partir des résultats** (aucun chiffre saisi à la main).

In [ ]:
tot_clv = out["Predicted_CLV"].sum()
hv = out[out["Customer_Value"] == "High CLV"]
ar = segsum.loc["At Risk"]
tot_clv = max(tot_clv, 1e-9)
n_active = int((out["Already_Churned"] == 0).sum())
print(f"• {len(out):,} clients analysés ; {out['Already_Churned'].mean()*100:.1f} % sont déjà inactifs depuis plus de {CHURN_DAYS} jours (churn observé).")
print(f"• Les {len(hv):,} clients 'High CLV' (top {int((1-CLV_HIGH_Q)*100)} %) concentrent {hv['Predicted_CLV'].sum()/tot_clv*100:.1f} % de la CLV espérée totale.")
print(f"• Segment At Risk : {int(ar['Clients']):,} clients, soit £{ar['CLV_totale']:,.0f} de CLV potentiellement menacée ({ar['% CLV totale']:.1f} % de la CLV totale).")
print(f"• Revenu prédit sur les {HORIZON} prochains jours : £{out['Future_Revenue'].sum():,.0f} (profit à {MARGIN:.0%} : £{MARGIN*out['Future_Revenue'].sum():,.0f}).")
print(f"• Modèle de revenu retenu : {BEST} (RMSE test £{res.loc[BEST,'RMSE']:,.0f} vs baseline £{res.loc['Baseline (moyenne)','RMSE']:,.0f} ; R² = {res.loc[BEST,'R2']:.3f}).")
print(f"• Survie : C-index Cox = {cindex_cox:.3f} ; C-index Weibull = {cindex_w:.3f} ; AUC du risque à 180 j = {auc:.3f} (objectif indicatif C-index > 0.60 : {'atteint' if max(cindex_cox, cindex_w) > 0.60 else 'NON atteint'}).")
print(f"• Churn médian KM : {kmf.median_survival_time_} jours ; S(180 j) = {float(kmf.predict(180)):.2f}.")
print("\nVariables Cox qui AUGMENTENT le risque :", ", ".join(sm[sm['coef'] > 0].sort_values('coef', ascending=False).index[:3]) or "—")
print("Variables Cox qui DIMINUENT le risque  :", ", ".join(sm[sm['coef'] < 0].sort_values('coef').index[:3]) or "—")

### Préparation du dashboard (Power BI)

On exporte 4 fichiers CSV à importer dans Power BI (*Obtenir des données → Texte/CSV*) :
- `customers_clv_segments.csv` : une ligne par client (table de faits) ;
- `kpis.csv` : KPI globaux ;
- `segment_summary.csv` : synthèse par segment ;
- `top_priority_customers.csv` : Top 50 clients à contacter.

**Filtres (segments / slicers) :** `Segment`, `Country`, `CLV_Level`, `Risk_Level`.
**Pages suggérées :** (1) Vue d'ensemble : cartes KPI + CLV par segment ; (2) Clients à risque : scatter Churn_Risk × Predicted_CLV + table triée par `Priority_Score` ; (3) Pays / segments.

In [ ]:
kpis = pd.DataFrame({
    "KPI": ["Total Customers", "Total Revenue (£, historique)", "Average CLV (£)", "Median CLV (£)", "High-Value Customers",
            "High-Risk Customers", "Churn Rate (Recency > 180 j)", "Average Customer Lifetime (jours)", "Future Revenue prédit 180 j (£)"],
    "Value": [len(out), df["Revenue"].sum(), out["Predicted_CLV"].mean(), out["Predicted_CLV"].median(),
              int((out["Customer_Value"] == "High CLV").sum()), int((out["Risk_Level"] == "High Risk").sum()),
              out["Already_Churned"].mean(), out["Lifetime"].mean(), out["Future_Revenue"].sum()]})
display(kpis)

export = out.reset_index().rename(columns={"index": "CustomerID"})
export["CustomerID"] = export["CustomerID"].astype(int)
export.to_csv("customers_clv_segments.csv", index=False)
kpis.to_csv("kpis.csv", index=False)
segsum.reset_index().to_csv("segment_summary.csv", index=False)
out[mask].sort_values("CLV_at_Risk", ascending=False).head(50).reset_index().to_csv("top_priority_customers.csv", index=False)
print("Fichiers exportés :", [f for f in os.listdir('.') if f.endswith('.csv')])
try:
    from google.colab import files
    for f in ["customers_clv_segments.csv", "kpis.csv", "segment_summary.csv", "top_priority_customers.csv"]:
        files.download(f)
except Exception:
    pass

## 23. Conclusion

**Ce que le projet a construit  :**
1. Nettoyage documenté (règles + lignes supprimées) et EDA : forte concentration de la valeur, saisonnalité, domination UK.
2. **Table client** et analyse **RFM** : profils de comportement.
3. **Split temporel** (observation → futur 180 j) : prédiction du revenu futur sans data leakage, modèles comparés à un baseline (MAE, RMSE, R²).
4. **Survival Analysis** : churn = 180 j d'inactivité (seuils 90/270 testés), censure à droite, Kaplan-Meier, Cox (hazard ratios, C-index) et Weibull (survie future).
5. **Predicted_CLV** = marge 30 % × revenu prédit × survie cumulée actualisée.
6. **Segmentation** Champions / Potential Loyalists / At Risk / Low Value, **Priority_Score** et Top 20 de clients à contacter.
7. Export CSV pour un **dashboard Power BI**.

**Limites à mentionner à l'oral :** un seul dataset (UK, 2 ans), marge fixe de 30 %, revenu supposé stable d'une période à l'autre, survie extrapolée au-delà de 365 j, seuils de segmentation arbitraires mais paramétrables, R² modeste attendu (futur incertain).

**Pistes d'amélioration :** modèles BG/NBD + Gamma-Gamma comme benchmark, validation temporelle glissante, variables saisonnières, coûts de campagne pour optimiser le ROI marketing.

**Phrase finale :** *À partir de l'historique d'achat des clients, nous prédisons leur revenu futur, estimons leur probabilité de rester actifs grâce à la Survival Analysis, calculons leur Customer Lifetime Value et les segmentons selon leur valeur et leur risque de churn afin de proposer des actions marketing adaptées.*